# Метки сделок из опросов

Берём ответы «Продал на Авито» и сопоставляем их с исходными объявлениями по `listing_id`. Для каждого найденного объявления ставим `is_deal = 1` и `weight = 0.8`. Запускать из корня репозитория.

## 1. Загрузить данные

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

SURVEYS_PATH = Path("data/surveys.csv")
LISTINGS_PATH = Path("data/listings.csv")
OUTPUT_PATH = Path("data_by_dewallen/survey_deal_labels.csv")

surveys = pd.read_csv(SURVEYS_PATH, usecols=["listing_id", "answer"])
listings = pd.read_csv(LISTINGS_PATH, usecols=["listing_id"])

## 2. Сопоставить ответы с объявлениями

In [2]:
positive_surveys = surveys.loc[surveys["answer"] == "Продал на Авито", ["listing_id"]]
positive_ids = positive_surveys.dropna().drop_duplicates()
listing_ids = listings.dropna().drop_duplicates()

matched = positive_ids.merge(
    listing_ids, on="listing_id", how="inner", validate="one_to_one",
)

print("Ответов «Продал на Авито»:", len(positive_surveys))
print("Уникальных ID в ответах:", len(positive_ids))
print("Объявлений после сопоставления:", len(matched))

Ответов «Продал на Авито»: 2531
Уникальных ID в ответах: 2480
Объявлений после сопоставления: 2480


## 3. Сохранить метки

In [3]:
labels = matched.sort_values("listing_id").assign(is_deal=1, weight=0.8)

assert labels.columns.tolist() == ["listing_id", "is_deal", "weight"]
assert labels["listing_id"].is_unique, "В результате повторяются ID объявлений"
assert set(labels["listing_id"]) == set(positive_ids["listing_id"]) & set(listing_ids["listing_id"])

OUTPUT_PATH.parent.mkdir(exist_ok=True)
labels.to_csv(OUTPUT_PATH, index=False)

print("Сохранено объявлений:", len(labels))
display(labels.head())

Сохранено объявлений: 2480


,listing_id,is_deal,weight
15,100013,1,0.8
32,100043,1,0.8
112,100051,1,0.8
2,100114,1,0.8
13,100157,1,0.8
